# 03 — Satellite View Segmentation

`POST /v1/satellite` classifies land cover (buildings, vegetation, pavement, etc.) at a tile centred on a point.

**Plan:** Premium only.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from dotenv import load_dotenv; load_dotenv(pathlib.Path.cwd().parent / '.env')

from fortyguard import FortyGuardClient
from fortyguard.samples import CHICAGO_POINT
client = FortyGuardClient()

In [ ]:
response = client.satellite_segmentation(
    latitude=CHICAGO_POINT['latitude'],
    longitude=CHICAGO_POINT['longitude'],
    start_date='2024-07-15',
    start_time='14:00',
    filter_type=1,
    granularity=80,
)
result = response['result']
print('Image year:', result.get('image_year'))
print('Segmentation keys:', list(result.get('segmentation', {}).keys()))

In [ ]:
import base64, io
from PIL import Image
import matplotlib.pyplot as plt

def _decode(b64):
    if not b64: return None
    if b64.startswith('data:'):
        b64 = b64.split(',', 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))

originals = result.get('orignal_image') or result.get('original_image') or []
seg_b64 = result.get('segmentation', {}).get('image_content')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
orig = _decode(originals[0]) if originals else None
mask = _decode(seg_b64)
for ax, img, title in zip(axes, [orig, mask], ['Original satellite tile', 'Segmentation mask']):
    if img is not None:
        ax.imshow(img)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Coverage percentages per class.
segments = result.get('segmentation', {}).get('segments', {})
for cls, pct in sorted(segments.items(), key=lambda kv: kv[1], reverse=True):
    print(f'  {cls:>30}: {pct}')